In [1]:
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 11

In [ ]:
RAW_FILE_PATH = "fifa_fbref_merged.csv"

In [ ]:
preview = pd.read_csv(RAW_FILE_PATH, nrows=5)
print("Total columns:", preview.shape[1])
preview.head()

In [ ]:
columns_to_use = [
    'short_name', 'player_positions', 'pos', 'age_fifa', 'height_cm', 'weight_kg',
    'overall', 'potential', 'value_eur', 'wage_eur',
    'club_name', 'league_name', 'nationality_name', 'preferred_foot', 'season',
    'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic',
    'Playing Time_Min', 'Playing Time_90s',
    'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A',
    'Per 90 Minutes_xG', 'Per 90 Minutes_xAG',
]

df = pd.read_csv(RAW_FILE_PATH, usecols=columns_to_use)
print("Shape:", df.shape)

memory_before_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f" memory usage before dtype opt : {memory_before_mb:,.1f} MB")
df.head()

In [ ]:
df.isna().sum()

In [ ]:
rows_before_na = len(df)

df = df.dropna(subset=[
    'short_name', 'pos', 'overall', 'value_eur', 'wage_eur', 'Playing Time_Min',
])

rows_after_na = len(df)
print(f"before : {rows_before_na:,}  |  after : {rows_after_na:,}  |  removed : {rows_before_na - rows_after_na:,}")

In [ ]:
rows_before = len(df)

MIN_MINUTES = 900

df = df[df['Playing Time_Min'] >= MIN_MINUTES]
df = df[(df['value_eur'] > 0) & (df['wage_eur'] > 0)]
df = df[df['overall'].between(1, 99)]

rows_after = len(df)
print(f" before: {rows_before:,}  |  after: {rows_after:,}  |  removed: {rows_before - rows_after:,} ({(rows_before - rows_after) / rows_before:.1%})")

In [ ]:
dtype_map = {
    'age_fifa': 'int8', 'height_cm': 'int16', 'weight_kg': 'int16',
    'overall': 'int8', 'potential': 'int8',
    'value_eur': 'float32', 'wage_eur': 'float32',
    'league_name': 'category', 'nationality_name': 'category',
    'preferred_foot': 'category', 'pos': 'category',
    'pace': 'float32', 'shooting': 'float32', 'passing': 'float32',
    'dribbling': 'float32', 'defending': 'float32', 'physic': 'float32',
    'Playing Time_Min': 'float32', 'Playing Time_90s': 'float32',
    'Per 90 Minutes_Gls': 'float32', 'Per 90 Minutes_Ast': 'float32',
    'Per 90 Minutes_G+A': 'float32', 'Per 90 Minutes_xG': 'float32',
    'Per 90 Minutes_xAG': 'float32',
}

for col, new_type in dtype_map.items():
    df[col] = df[col].astype(new_type)

memory_after_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f" memory usage before opt : {memory_before_mb:,.1f} MB")
print(f" memory usage after opt : {memory_after_mb:,.1f} MB")
print(f" memory usage reduction : {(1 - memory_after_mb / memory_before_mb):.1%}")

df.info(memory_usage='deep')

In [ ]:
# تحلیل رابطه رتبه فیفا با عملکرد واقعی

fig, ax = plt.subplots(figsize=(10, 6.5), facecolor='white')

hb = ax.hexbin(df['overall'], df['Per 90 Minutes_G+A'], gridsize=28,
                cmap='YlOrRd', mincnt=1, bins='log', edgecolors='#dddddd', linewidths=0.2)

cb = fig.colorbar(hb, ax=ax)
cb.set_label('Number of Players (log scale)', fontsize=10)

ax.set_title('FIFA Overall Rating vs. Real Goal Contributions', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('FIFA Overall Rating', fontsize=11)
ax.set_ylabel('Goals + Assists per 90 Minutes', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

correlation = df['overall'].corr(df['Per 90 Minutes_G+A'])
print(f"Correlation coefficient: {correlation:.2f}")

In [ ]:
# تحلیل رابطه دستمزد هفتگی و ارزش بازار

fig, ax = plt.subplots(figsize=(10, 6.5), facecolor='white')

scatter = ax.scatter(df['wage_eur'], df['value_eur'],
                      c=df['overall'], cmap='viridis', alpha=0.35, s=14,
                      edgecolors='none')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Weekly Wage vs. Market Value', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Weekly Wage (EUR, log scale)', fontsize=11)
ax.set_ylabel('Market Value (EUR, log scale)', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

cb = fig.colorbar(scatter, ax=ax)
cb.set_label('FIFA Overall Rating', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# تحلیل میانگین رتبه فیفا بازیکنان بر اساس لیگ

top_leagues = df['league_name'].value_counts().head(15).index
league_avg = df[df['league_name'].isin(top_leagues)].groupby('league_name', observed=True)['overall'].mean().sort_values()

fig, ax = plt.subplots(figsize=(9, 7), facecolor='white')

norm_values = (league_avg.values - league_avg.min()) / (league_avg.max() - league_avg.min())
bars = ax.barh(league_avg.index.astype(str), league_avg.values,
                color=plt.cm.YlOrRd(0.3 + 0.6 * norm_values), edgecolor='#555555', linewidth=0.4)

for bar, value in zip(bars, league_avg.values):
    ax.annotate(f'{value:.1f}', xy=(value, bar.get_y() + bar.get_height() / 2),
                xytext=(4, 0), textcoords='offset points', va='center', fontsize=9)

ax.set_title('Average FIFA Overall Rating by League (Top 15)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Average Overall Rating', fontsize=11)
ax.set_ylabel('')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()